# Часть 1: Материалы

In [3]:
!pip install biopython requests pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.1/809.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.1/754.1 kB 44.4 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.3-py3-none-any.whl size=43549 sha256=2ad7d071b96ae73e57aba02b2c473ec75ff3b3975d69a3ef4846673e4a7a9e65
  Stored in directory: /root/.cache/pip/wheels/1f/7d/e9/1ff2509f13767a55df1279744adfb757f4ab94b2cbe761f56a
Successfully built bibtexparser


In [4]:
from pymatgen.ext.matproj import MPRester
from pymatgen.core import Composition
# from pymatgen.core.composition import Composition
import requests
from Bio import SeqIO
from Bio.Seq import Seq
import io
import csv
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import itertools

In [5]:
API_KEY = "9Ks9EkHBf4ICfALQavd5ItjrzC5e2q0G"

1. Структура материала

In [15]:
mpr = MPRester(API_KEY)
structures = mpr.get_structures("MnFe2O4")
structure = structures[0]

elements_sites = {}
for site in structure:
    element = site.species_string
    if element not in elements_sites:
        elements_sites[element] = site

print("Первый PeriodicSite для каждого элемента:")
for element, site in elements_sites.items():
    print(f"{element}: {site}")

Первый PeriodicSite для каждого элемента:
Mn: [0.90333746 0.63827435 1.52745529] Mn
Fe: [2.69321975e+00 6.92801017e-04 4.55987516e+00] Fe
O: [1.89600419 1.36518036 3.21058204] O


2. Анализ состава и элементов материала

In [16]:
print("=== Анализ состава материала ===")
composition = structure.composition

print("Формула:", composition.formula)
print("\nЭлементы и их количество (шт.):")
for el, amt in composition.get_el_amt_dict().items():
    print(f"  {el}: {amt}")

print("\nАтомная доля элементов (%):")
total_atoms = sum(composition.get_el_amt_dict().values())
for el, amt in composition.get_el_amt_dict().items():
    frac = (amt / total_atoms) * 100
    print(f"  {el}: {frac:.2f}%")

print("\nМолекулярная масса материала:")
print(f"  {composition.weight:.4f} г/моль")


=== Анализ состава материала ===
Формула: Mn10 Fe20 O40

Элементы и их количество (шт.):
  Mn: 10.0
  Fe: 20.0
  O: 40.0

Атомная доля элементов (%):
  Mn: 14.29%
  Fe: 28.57%
  O: 57.14%

Молекулярная масса материала:
  2306.2565 г/моль


In [17]:
print("\n=== Свойства элементов ===")
for el in composition.elements:
    print(f"Элемент: {el}")
    print(f"  Атомный номер: {el.Z}")
    print(f"  Атомная масса: {el.atomic_mass:.4f} а.е.м.")
    print(f"  Группа: {el.group}")
    print(f"  Радиус Ван-дер-Ваальса: {el.van_der_waals_radius}")
    print(f"  Электроотрицательность: {el.X}")


=== Свойства элементов ===
Элемент: Mn
  Атомный номер: 25
  Атомная масса: 54.9380 а.е.м.
  Группа: 7
  Радиус Ван-дер-Ваальса: 2.05 ang
  Электроотрицательность: 1.55
Элемент: Fe
  Атомный номер: 26
  Атомная масса: 55.8450 а.е.м.
  Группа: 8
  Радиус Ван-дер-Ваальса: 2.04 ang
  Электроотрицательность: 1.83
Элемент: O
  Атомный номер: 8
  Атомная масса: 15.9994 а.е.м.
  Группа: 16
  Радиус Ван-дер-Ваальса: 1.52 ang
  Электроотрицательность: 3.44


3. Параметры решетки и плотность

In [18]:
lattice = structure.lattice
print("\n=== Параметры решётки ===")
print(f"a = {lattice.a:.4f} Å")
print(f"b = {lattice.b:.4f} Å")
print(f"c = {lattice.c:.4f} Å")
print(f"α = {lattice.alpha:.2f}°")
print(f"β = {lattice.beta:.2f}°")
print(f"γ = {lattice.gamma:.2f}°")
print(f"Объём ячейки = {lattice.volume:.4f} Å³")


=== Параметры решётки ===
a = 6.1505 Å
b = 6.1007 Å
c = 30.3960 Å
α = 60.28°
β = 60.00°
γ = 59.84°
Объём ячейки = 807.1544 Å³


In [19]:
density = structure.density
print("\n=== Плотность материала ===")
print(f"Плотность: {density:.4f} г/см³")


=== Плотность материала ===
Плотность: 4.7446 г/см³


# Часть 2: Последовательности ДНК/РНК

1. Создание произвольных последовательностей

In [34]:
import random
# На самом деле не знаю можно ли так делать, но попробую
def generate_random_dna(length):
    """Генерирует случайную ДНК последовательность заданной длины"""
    bases = ['A', 'T', 'C', 'G']
    return Seq(''.join(random.choice(bases) for _ in range(length)))

# Генерируем случайную последовательность длиной 20 нуклеотидов
random_dna1, random_dna2, random_dna3  = generate_random_dna(20), generate_random_dna(20), generate_random_dna(20)
print(f'Последовательность ДНК 1: {random_dna1},\nПоследовательность ДНК 2: {random_dna2},\nПоследовательность ДНК 3: {random_dna3}')

Последовательность ДНК 1: CTAGACTTGCGAGCCGGGAA,
Последовательность ДНК 2: ATTCAATTATGCGGGTTCAG,
Последовательность ДНК 3: TGGAGAAATGCCCAGCCGGC


2. GC-состав

In [35]:
from Bio.SeqUtils import gc_fraction

sequences = [random_dna1, random_dna2, random_dna3]

for i, seq_obj in enumerate(sequences):
    gc_content = gc_fraction(seq_obj)
    print(f"Последовательность ДНК {i+1}: {seq_obj} - GC-состав: {gc_content:.2f}%")

Последовательность ДНК 1: CTAGACTTGCGAGCCGGGAA - GC-состав: 0.60%
Последовательность ДНК 2: ATTCAATTATGCGGGTTCAG - GC-состав: 0.40%
Последовательность ДНК 3: TGGAGAAATGCCCAGCCGGC - GC-состав: 0.65%


3. Обратные комплементарные последовательности и транскрибация в РНК

In [37]:
reversed_seq = []
for i, seq_obj in enumerate(sequences):
  reverse_complement_seq = seq_obj.reverse_complement()
  reversed_seq.append(reverse_complement_seq)
  print(f'Обратная комплементарная последовательность {i+1}: {reverse_complement_seq}')

Обратная комплементарная последовательность 1: TTCCCGGCTCGCAAGTCTAG
Обратная комплементарная последовательность 2: CTGAACCCGCATAATTGAAT
Обратная комплементарная последовательность 3: GCCGGCTGGGCATTTCTCCA


In [39]:
# Как я поняла, нужно транскрибировать комплиментарную ДНК
for i, seq_obj in enumerate(reversed_seq):
  transcribed_seq = seq_obj.transcribe()
  print(f'Транскрибация в РНК для последовательности {i+1}: {transcribed_seq}')

Транскрибация в РНК для последовательности 1: UUCCCGGCUCGCAAGUCUAG
Транскрибация в РНК для последовательности 2: CUGAACCCGCAUAAUUGAAU
Транскрибация в РНК для последовательности 3: GCCGGCUGGGCAUUUCUCCA


# Часть 3: Белки

1. Последовательность аминокислот

UniProt ID для человеческого белка Альфа Амилаза: [P0DTE8](https://www.uniprot.org/uniprotkb/P0DTE8/entry)

In [45]:
from Bio import ExPASy
from Bio import SwissProt

ID = "P0DTE8"

# Получаем данные из UniProt
handle = ExPASy.get_sprot_raw(ID)
record = SwissProt.read(handle)
handle.close()

sequence = record.sequence

print('Последовательность аминокислот:')
for i in range(0, len(sequence), 60):
    print(sequence[i:i+60])

Последовательность аминокислот:
MKLFWLLFTIGFCWAQYSSNTQQGRTSIVHLFEWRWVDIALECERYLAPKGFGGVQVSPP
NENVAIHNPFRPWWERYQPVSYKLCTRSGNEDEFRNMVTRCNNVGVRIYVDAVINHMCGN
AVSAGTSSTCGSYFNPGSRDFPAVPYSGWDFNDGKCKTGSGDIENYNDATQVRDCRLSGL
LDLALGKDYVRSKIAEYMNHLIDIGVAGFRIDASKHMWPGDIKAILDKLHNLNSNWFPEG
SKPFIYQEVIDLGGEPIKSSDYFGNGRVTEFKYGAKLGTVIRKWNGEKMSYLKNWGEGWG
FMPSDRALVFVDNHDNQRGHGAGGASILTFWDARLYKMAVGFMLAHPYGFTRVMSSYRWP
RYFENGKDVNDWVGPPNDNGVTKEVTINPDTTCGNDWVCEHRWRQIRNMVNFRNVVDGQP
FTNWYDNGSNQVAFGRGNRGFIVFNNDDWTFSLTLQTGLPAGTYCDVISGDKINGNCTGI
KIYVSDDGKAHFSISNSAEDPFIAIHAESKL


2. Физико-химические дескрипторы белка (состав аминокислот, частоты дипептидов)

In [59]:
total = len(sequence)
analysed_seq = ProteinAnalysis(str(sequence))

amino_acid_comp = analysed_seq.count_amino_acids()
print("Состав аминокислот:")
for amino in amino_acid_comp.items():
  print(f' Аминокислота {amino[0]}: {amino[1]} ({amino[1]/total*100:.2f}%)')

Состав аминокислот:
 Аминокислота A: 27 (5.28%)
 Аминокислота C: 12 (2.35%)
 Аминокислота D: 35 (6.85%)
 Аминокислота E: 20 (3.91%)
 Аминокислота F: 29 (5.68%)
 Аминокислота G: 52 (10.18%)
 Аминокислота H: 12 (2.35%)
 Аминокислота I: 28 (5.48%)
 Аминокислота K: 24 (4.70%)
 Аминокислота L: 27 (5.28%)
 Аминокислота M: 11 (2.15%)
 Аминокислота N: 41 (8.02%)
 Аминокислота P: 22 (4.31%)
 Аминокислота Q: 12 (2.35%)
 Аминокислота R: 28 (5.48%)
 Аминокислота S: 33 (6.46%)
 Аминокислота T: 23 (4.50%)
 Аминокислота V: 35 (6.85%)
 Аминокислота W: 19 (3.72%)
 Аминокислота Y: 21 (4.11%)


В библиотеке BioPython я не нашла метода подсчёта частоты бипептитов, поэтому реализовала это вручную, но результаты странные, буду рада обратной связи насчёт того что я неверно сделала

In [60]:
from collections import Counter
dipeptides = [sequence[i:i+2] for i in range(len(sequence)-1)]

dipeptide_counts = Counter(dipeptides)
total_dipeptides = len(dipeptides)
dipeptide_frequencies = {dipeptide: count/total_dipeptides for dipeptide, count in dipeptide_counts.items()}

print("Частоты дипептидов:")
for dipeptide, freq in sorted(dipeptide_frequencies.items()):
    print(f"{dipeptide}: {freq:.2f} %")

Частоты дипептидов:
AE: 0.01
AF: 0.00
AG: 0.01
AH: 0.00
AI: 0.01
AK: 0.00
AL: 0.01
AP: 0.00
AQ: 0.00
AR: 0.00
AS: 0.00
AT: 0.00
AV: 0.01
CD: 0.00
CE: 0.00
CG: 0.01
CK: 0.00
CN: 0.00
CR: 0.00
CT: 0.00
CW: 0.00
DA: 0.01
DC: 0.00
DD: 0.00
DE: 0.00
DF: 0.00
DG: 0.01
DI: 0.01
DK: 0.00
DL: 0.00
DN: 0.01
DP: 0.00
DR: 0.00
DT: 0.00
DV: 0.00
DW: 0.01
DY: 0.00
EC: 0.00
ED: 0.00
EF: 0.00
EG: 0.00
EH: 0.00
EK: 0.00
EN: 0.01
EP: 0.00
ER: 0.00
ES: 0.00
EV: 0.00
EW: 0.00
EY: 0.00
FC: 0.00
FE: 0.00
FG: 0.01
FI: 0.01
FK: 0.00
FM: 0.00
FN: 0.01
FP: 0.00
FR: 0.01
FS: 0.00
FT: 0.01
FV: 0.00
FW: 0.00
GA: 0.01
GD: 0.01
GE: 0.01
GF: 0.01
GG: 0.01
GH: 0.00
GI: 0.00
GK: 0.01
GL: 0.00
GN: 0.01
GP: 0.00
GQ: 0.00
GR: 0.01
GS: 0.01
GT: 0.01
GV: 0.01
GW: 0.00
HA: 0.00
HD: 0.00
HF: 0.00
HG: 0.00
HL: 0.00
HM: 0.00
HN: 0.00
HP: 0.00
HR: 0.00
IA: 0.01
ID: 0.01
IE: 0.00
IG: 0.00
IH: 0.00
IK: 0.01
IL: 0.00
IN: 0.01
IR: 0.00
IS: 0.00
IV: 0.00
IY: 0.01
KA: 0.00
KC: 0.00
KD: 0.00
KE: 0.00
KG: 0.00
KH: 0.00
KI: 0.01
KL: 0.01